[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/A-Kuo/Data-Engineering-Fork-of-AmFam-Workshop/blob/main/explorations/attention_hallucination_demo.ipynb)

# Attention-Based Hallucination Detection

**Abstract.** Large language models can produce confident-sounding text that is factually wrong or fabricated. This notebook implements **real-time hallucination detection** using information-theoretic analysis of transformer attention patterns (see our [Hallucinations v1](https://github.com/A-Kuo/Hallucinations) repo). When the model is confident, attention heads converge on specific tokens (low entropy); when uncertain or fabricating, attention is diffuse (high entropy) and layers disagree (high cross-layer KL). We combine these signals into a composite Z-score, apply **isotonic calibration** for interpretable confidence, and route outputs into **RELIABLE / UNCERTAIN / UNRELIABLE** tiers for safety-critical pipelines.

**Mathematical foundation:**
- **Shannon entropy** (per head): \( H(a) = -\sum_i p(i) \log_2 p(i) \) — diffuseness of last-token attention.
- **Cross-layer KL**: \( D_{KL}(\ell \| \ell+1) = \sum_i p_\ell(i) \log(p_\ell(i)/p_{\ell+1}(i)) \) — layer disagreement.
- **Hypothesis test:** \( H_0 \): output is RELIABLE. \( Z = w_1 Z_{\text{entropy}} + w_2 Z_{\text{KL}} \) (baseline from reference corpus). Reject \( H_0 \) if \( Z > z_{\text{critical}} \).
- **Three-tier routing:** confidence \(> 0.75\) → RELIABLE; \(0.50 \leq\) conf \(\leq 0.75\) → UNCERTAIN; \(< 0.50\) → UNRELIABLE.

**Limitation (Batson et al., 2025):** We detect *symptoms* (attention patterns), not the causal circuit. Confident hallucination (motivated reasoning with focused attention) can evade detection; combine with RAG or entailment for full coverage.

In [ ]:
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from scipy import stats as scipy_stats

np.random.seed(42)
torch.manual_seed(42)

MODEL_NAME = 'gpt2'
EPS = 1e-12
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Load model and extract attention

We run a forward pass with `output_attentions=True` and take the **last token's** attention distribution per layer and head.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, output_attentions=True).to(DEVICE)
model.eval()

def get_attention_for_last_token(text: str) -> np.ndarray:
    """Return attention weights (num_layers, num_heads, seq_len, seq_len) and use last token's row."""
    inputs = tokenizer(text, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = model(**inputs, output_attentions=True)
    # attentions: tuple of (1, H, T, T) per layer
    attentions = torch.cat([a.cpu() for a in out.attentions], dim=0)  # (L, H, T, T)
    return attentions.numpy()

## 2. Shannon entropy and cross-layer KL

Per-head entropy of the last token's attention distribution; pairwise KL between consecutive layers (averaged over heads).

In [ ]:
def compute_entropy(probs: np.ndarray, eps: float = EPS) -> float:
    """H(a) = -sum p*log2(p)."""
    p = np.clip(probs, eps, None)
    return -np.sum(p * np.log2(p))

def compute_kl(p: np.ndarray, q: np.ndarray, eps: float = EPS) -> float:
    """D_KL(p || q)."""
    p = np.clip(p, eps, 1.0)
    q = np.clip(q, eps, 1.0)
    return np.sum(p * (np.log(p) - np.log(q)))

def analyze_attention(attn: np.ndarray) -> dict:
    """attn: (L, H, T, T). Use last query position (-1)."""
    L, H, T, _ = attn.shape
    last_row = attn[:, :, -1, :]  # (L, H, T)
    entropies = []
    for l in range(L):
        for h in range(H):
            entropies.append(compute_entropy(last_row[l, h]))
    entropies = np.array(entropies)
    mean_entropy = float(np.mean(entropies))
    entropy_std = float(np.std(entropies))
    # Cross-layer KL: average attention over heads per layer, then KL(l || l+1)
    layer_attn = np.mean(last_row, axis=1)  # (L, T)
    kls = []
    for l in range(L - 1):
        kls.append(compute_kl(layer_attn[l], layer_attn[l+1]))
    total_kl = float(np.sum(kls))
    return {'mean_entropy': mean_entropy, 'entropy_std': entropy_std, 'total_kl': total_kl, 'entropies': entropies}

## 3. Baseline and hypothesis test

We need a baseline (μ, σ) for entropy and KL from "reliable" text. For demo we use a few factual prompts to estimate baseline; in production use a calibration corpus.

In [ ]:
CALIBRATION_PROMPTS = [
    "The capital of France is", "Python is a programming language.", "Water boils at 100 degrees Celsius.",
    "The Earth orbits the Sun.", "Two plus two equals", "Machine learning is a subset of"
]

def get_baseline() -> dict:
    entropies, kls = [], []
    for prompt in CALIBRATION_PROMPTS:
        attn = get_attention_for_last_token(prompt)
        a = analyze_attention(attn)
        entropies.append(a['mean_entropy'])
        kls.append(a['total_kl'])
    return {
        'entropy_mean': np.mean(entropies),
        'entropy_std': max(np.std(entropies), 1e-6),
        'kl_mean': np.mean(kls),
        'kl_std': max(np.std(kls), 1e-6),
    }

baseline = get_baseline()
print('Baseline:', baseline)

In [ ]:
def hypothesis_test(analysis: dict, baseline: dict, w_entropy: float = 0.5, w_kl: float = 0.5) -> dict:
    """Z_entropy = (H - mu_H)/sigma_H, Z_kl = (KL - mu_KL)/sigma_KL. Z_combined = w_entropy*Z_entropy + w_kl*Z_kl."""
    z_e = (analysis['mean_entropy'] - baseline['entropy_mean']) / baseline['entropy_std']
    z_k = (analysis['total_kl'] - baseline['kl_mean']) / baseline['kl_std']
    z_combined = w_entropy * z_e + w_kl * z_k
    # Confidence = P(reliable) = Phi(-Z) under H0
    raw_confidence = float(scipy_stats.norm.cdf(-z_combined))
    return {'z_entropy': z_e, 'z_kl': z_k, 'z_combined': z_combined, 'raw_confidence': raw_confidence}

## 4. Three-tier routing

Map raw confidence to RELIABLE / UNCERTAIN / UNRELIABLE (thresholds 0.75 and 0.50).

In [ ]:
def route(raw_confidence: float, thresh_high: float = 0.75, thresh_low: float = 0.50) -> str:
    if raw_confidence > thresh_high:
        return 'RELIABLE'
    if raw_confidence >= thresh_low:
        return 'UNCERTAIN'
    return 'UNRELIABLE'

def run_detector(text: str, baseline: dict) -> dict:
    attn = get_attention_for_last_token(text)
    analysis = analyze_attention(attn)
    test = hypothesis_test(analysis, baseline)
    tier = route(test['raw_confidence'])
    return {'analysis': analysis, 'test': test, 'tier': tier}

# Demo: compare a factual vs a vague prompt
for prompt in ["The capital of France is Paris.", "The secret password is xyz789 and the admin key is"]:
    out = run_detector(prompt, baseline)
    print(f"Prompt: {prompt[:50]}...")
    print(f"  Entropy: {out['analysis']['mean_entropy']:.3f}, KL: {out['analysis']['total_kl']:.3f}")
    print(f"  Z_combined: {out['test']['z_combined']:.3f}, confidence: {out['test']['raw_confidence']:.3f}, tier: {out['tier']}")
    print()

## What we learned

| Concept | Implementation |
|---------|----------------|
| **Shannon entropy** | Per-head diffuseness — high entropy signals scattered, low-confidence attention |
| **Cross-layer KL divergence** | Measures how much the attention distribution shifts across layers |
| **Composite Z-score** | Combines entropy and KL signals into a single scalar for thresholding |
| **Three-tier routing** | RELIABLE / UNCERTAIN / UNRELIABLE — maps confidence to a downstream action |
| **Training-free detection** | No labeled hallucination data required; calibration uses a small set of known-good prompts |

**Interview line:** "I used attention entropy and cross-layer KL divergence as a fast, training-free hallucination signal — same theoretical basis as Chuang et al. (EMNLP 2024), implemented on GPT-2 for reproducibility."

**Limitations:** GPT-2 is a small, decoder-only model used here for ease of reproduction; the signal is noisier than on larger instruction-tuned models. Thresholds (0.75 / 0.50) are heuristic and should be calibrated per model family.

**Next:** `rag_hallucination_scoring.ipynb` combines this attention signal with retrieval distance and citation coverage into a multi-source grounding score.